# 🌊 Sonaris — Multi-Class Lite Model Training
### Train YOLO11 on the 4-Class Seabed Sonar Dataset

This notebook trains the **Sonaris Lite Model** on the **SeabedObjects-KLSG** dataset with **4 classes**:
- `0: aircraft` (Downed planes / fuselage debris on seabed)
- `1: fish` (Acoustic fish swarms)
- `2: other` (Seabed debris, man-made structures, pipelines, containers)
- `3: shipwreck` (Sunken vessels and hulls)

⏱️ **Estimated Training Time**: ~15-20 minutes on Colab Free T4 GPU.

## Step 1: Install Dependencies & Check GPU

In [ ]:
!pip install ultralytics roboflow -q

import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ Please enable GPU: Runtime -> Change runtime type -> T4 GPU")

## Step 2: Download the 4-Class Sonar Dataset from Roboflow Universe

In [ ]:
# Download the public SeabedObjects-KLSG sonar dataset (654 images)
from roboflow import Roboflow

# Public dataset on Roboflow Universe
# You can sign in to roboflow.com (free) to get your API key, or use curl:
!curl -L "https://universe.roboflow.com/ds/Z04L0pUaB4?key=your_key_or_download" > sonar_dataset.zip || echo "Alternative download below"

# Or download using Python API:
# rf = Roboflow(api_key="YOUR_API_KEY")
# project = rf.workspace("object-detect-ury2h").project("sonar_detect")
# dataset = project.version(1).download("yolov11")

## Step 3: Train YOLO11 Nano (Lite Model)

In [ ]:
from ultralytics import YOLO

# Load lightweight base YOLO11 model
model = YOLO('yolo11n.pt')

# Train on the 4-class sonar dataset
results = model.train(
    data='sonar_detect-1/data.yaml',   # Path to downloaded data.yaml
    epochs=100,                        # 100 epochs for high convergence
    imgsz=640,                         # 640 or 1024 for sonar detail
    batch=16,
    patience=25,                       # Early stopping if no improvement
    device=0,                          # GPU acceleration
    optimizer='AdamW',
    lr0=0.001,
    weight_decay=0.001,
    cos_lr=True,
    augment=True,
    mosaic=0.5,
    project='sonaris_runs',
    name='sonaris_multiclass_lite',
    exist_ok=True
)

print("Training completed!")

## Step 4: Evaluate the 4 Classes

In [ ]:
# Validate on the test split
metrics = model.val(data='sonar_detect-1/data.yaml', split='test')

print("\n--- 📊 Final Validation Metrics ---")
print(f"Overall mAP@50:    {metrics.box.map50:.3f}")
print(f"Overall mAP@50-95: {metrics.box.map:.3f}")

# Display per-class AP50
for i, cls_name in enumerate(model.names.values()):
    if i < len(metrics.box.maps):
        print(f"  • Class '{cls_name}': AP@50 = {metrics.box.maps[i]:.3f}")

## Step 5: Download the Trained Model Weights
Download `best.pt` and replace `Sonaris/models/yolo11n_seg_best.pt` on your computer!

In [ ]:
from google.colab import files
files.download('sonaris_runs/sonaris_multiclass_lite/weights/best.pt')